# qCycleGAN Training Tutorial

## What is qCycleGAN?

**qCycleGAN** is a hybrid quantum-classical image translation model that combines:
- **Classical component**: A diffusion-based image translation network (CycleGAN-Turbo) trained with LoRA adapters, from https://github.com/GaParmar/img2img-turbo
- **Quantum component**: A variational quantum circuit (boson sampler) that processes intermediate image embeddings to provide additional annotations to the text annotations

### Why Quantum?
The quantum encoder learns to map high-dimensional classical embeddings into a quantum-accessible space and back, potentially discovering latent patterns that are hard to find classically.

### The Training Pipeline
This notebook walks you through the complete training workflow:
1. ✅ Load & validate configurations from YAML
2. ✅ Optionally override hyperparameters for experimentation
3. ✅ Launch the unified training loop
4. ✅ Evaluate checkpoints with FID/DINO metrics

This notebook should guide you into how to configure, run, and evaluate qCycleGAN (maybe for your own image translation tasks !)




## Part 1: Setup & Prerequisites

Before we dive in, let's ensure your environment is ready.

### Requirements
- **Working directory**: You should be in the project root (`CycleGAN-Turbo/`)
- **Python environment**: Dependencies from `requirements.txt` installed
- **Accelerate configuration**: Run `accelerate config` once (handles multi-GPU setup)

### Why Config-Driven?
The refactored pipeline uses **dataclass-based configuration** (Python 3.10+ feature):
- All hyperparameters live in YAML files (`config/experiments/`)
- Configurations are strongly-typed with validation
- Easy to reproduce experiments and share settings
- Same config works for both CLI (`train.py`) and notebook workflows

### About the amplitude encoding in the Quantum Encoder (or Boson Sampler)
- The quantum encoder now performs **amplitude encoding with automatic padding** to match the computation space
- A new training flag `training.unet_trained` lets you decide whether to fine-tune the full UNet or only its LoRA adapters. If it is trained fully, it allows a better adaption to the quantum annotations. We'll surface it later in this notebook so you can experiment with both regimes.

In [ ]:
# Install dependencies if needed (comment out if already done)
# %pip install -r requirements.txt

# First, verify we can import key modules
print("Checking environment...")
try:
    import torch
    import yaml
    from config.loader import load_configs
    print(f"✓ PyTorch {torch.__version__}")
    print(f"✓ YAML support available")
    print(f"✓ Config system ready")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Run: %pip install -r requirements.txt")

## Part 2: Understanding the Configuration System

### How Configuration Works

The training pipeline is controlled by three interconnected config objects:

```
quantum_cfg          dataset_cfg          training_cfg
├─ num_modes        ├─ dataset_folder   ├─ learning_rate
├─ num_photons      ├─ batch_size       ├─ max_train_steps
├─ trainable_params ├─ transform_type   ├─ lambda_cycle (loss weight)
└─ quantum (enable) └─ prompts          └─ validation_steps
```

All three are **loaded from a single YAML file**, ensuring consistency across classical and quantum components.

### Load & Inspect the Base Configuration

Let's load the default experiment configuration and see what we're working with:

In [ ]:
from pathlib import Path
from pprint import pprint
from config.loader import load_configs

# Path to the default experiment configuration
BASE_CONFIG = Path('src_quantum/config/experiments/quantum_pretrained.yaml')
EXPERIMENT_CONFIG = BASE_CONFIG

# Load the three config objects
quantum_cfg, dataset_cfg, training_cfg, classical_cfg = load_configs(EXPERIMENT_CONFIG)

print("=" * 60)
print("QUANTUM CONFIG (Boson Sampler Settings)")
print("=" * 60)
print(vars(quantum_cfg))

print("\n" + "=" * 60)
print("DATASET CONFIG (Data & Preprocessing)")
print("=" * 60)
print(vars(dataset_cfg))

print("\n" + "=" * 60)
print("TRAINING CONFIG (Training Loop & Hyperparameters)")
print("=" * 60)
print(vars(training_cfg))

## Part 3: Customizing Configuration for Your Experiment

### Why Override?

Before training on your full dataset, you might want to:
- **Run a quick smoke test** with fewer steps
- **Test on a subset** of validation images
- **Change the output directory** for different experiments
- **Adjust quantum settings** (e.g., disable quantum mode for classical-only baseline)
- **Toggle full-UNet training** via the new `training.unet_trained` flag

### How It Works

The `replace()` function from Python's `dataclasses` module creates a new config object with selective overrides. We then serialize it back to YAML so the training script can read it—this ensures both CLI and notebook workflows use identical configuration.

We'll also surface the `unet_trained` option below so you can decide whether to update all UNet weights or stick to LoRA-only fine-tuning.

In [ ]:
# Example: Create a demo config with reduced scope (for quick testing)
from dataclasses import asdict, replace
import yaml

print("Creating custom configuration for demo/testing...")

# Make a copy with lighter settings for faster iteration
custom_training_cfg = replace(
    training_cfg,
    max_train_steps=500,              # Stop after 500 steps instead of full training
    validation_steps=100,             # Validate more frequently
    validation_num_images=10,         # Use only 10 images for validation (faster FID/DINO)
    output_dir='output/qcyclegan_demo',
    unet_trained=True,                # You can fine-tune the entire UNet for this run
)

# Serialize to a new YAML file
CUSTOM_CONFIG_PATH = Path('src_quantum/config/experiments/quantum_demo.yaml')
with CUSTOM_CONFIG_PATH.open('w') as fp:
    payload = {
        'quantum': asdict(quantum_cfg),
        'dataset': asdict(dataset_cfg),
        'training': asdict(custom_training_cfg),
    }
    yaml.safe_dump(payload, fp, sort_keys=False)

print(f"✓ Demo config saved to: {CUSTOM_CONFIG_PATH}")
print(f"Key changes for demo:")
print(f"  - Max steps: {training_cfg.max_train_steps} → {custom_training_cfg.max_train_steps}")
print(f"  - Validation images: {training_cfg.validation_num_images} → {custom_training_cfg.validation_num_images}")
print(f"  - Output: {training_cfg.output_dir} → {custom_training_cfg.output_dir}")
print(f"  - UNet training mode: {'FULL' if custom_training_cfg.unet_trained else 'LoRA-only'}")

# Update for next cells
EXPERIMENT_CONFIG = CUSTOM_CONFIG_PATH
training_cfg = custom_training_cfg

## Bonus: Previewing the Quantum Encoder Modes

The refactor splits our quantum encoder into two drop-in options:

- **Sorted encoding (default)** — mirrors the legacy qCycleGAN implementation and keeps compatibility with the historical checkpoints housed in `models/legacy/`.
- **Unsorted encoding** — powered by `models/quantum_encoder_unsorted.py` and Merlin's `LexGrouping`. It keeps the same API but explores a different Hilbert space ordering.

Both are toggled via `QuantumConfig.sort_encoding`. The next cell runs the unsorted Boson sampler directly from this notebook so you can confirm the dependencies (Perceval/Merlin) are wired up before launching long experiments.

In [ ]:
import torch
from dataclasses import replace
from models.quantum_encoder import BosonSampler

print("Instantiating unsorted Boson sampler for a quick smoke test...")
unsorted_cfg = replace(quantum_cfg, sort_encoding=False)
unsorted_dims = (1,) + tuple(dataset_cfg.quantum_dims)
print(f"Dims: {unsorted_dims} | Modes: {unsorted_cfg.num_modes} | Photons: {unsorted_cfg.num_photons}")

unsorted_sampler = BosonSampler(unsorted_dims, config=unsorted_cfg)
demo_batch = torch.rand(*unsorted_dims)
with torch.inference_mode():
    preview = unsorted_sampler(demo_batch)

print(f"Output shape: {tuple(preview.shape)}")
print(f"Value range: min={float(preview.min()):.4f}, max={float(preview.max()):.4f}")

## Part 4: Launching the Training Loop

### What Happens Inside `train.py`?

When we run `train.py` with a config file, it:

1. **Loads configs** from YAML → validates all settings
2. **Builds models** → classical U-Net + VAE adapters, quantum boson sampler
3. **Sets up data** → creates dataset and dataloader
4. **Initializes training** → optimizers, loss functions, callbacks
5. **Trains** → loops through epochs/steps with gradient updates
6. **Validates** → periodically computes FID and DINO metrics
7. **Saves checkpoints** → LoRA weights + quantum parameters

### Multi-GPU Training (Optional)

The Accelerate library handles device management automatically. To use multiple GPUs:

```bash
accelerate launch --num_processes=2 src_quantum/train.py --experiment_config <path>
```

For now, we'll use the default single-device setup.

### Launch Training

In [ ]:
import subprocess
import sys

# Build the command
run_args = [
    sys.executable,
    'src_quantum/train.py',
    '--experiment_config',
    str(EXPERIMENT_CONFIG),
]

print("=" * 60)
print("LAUNCHING TRAINING")
print("=" * 60)
print(f"Config: {EXPERIMENT_CONFIG}")
print(f"Output dir: {training_cfg.output_dir}")
print(f"Max steps: {training_cfg.max_train_steps}")
print(f"Quantum mode: {'ENABLED' if quantum_cfg.quantum else 'DISABLED'}")
print(f"UNet training: {'FULL NETWORK' if training_cfg.unet_trained else 'LoRA ADAPTERS ONLY'}")
print("=" * 60)
print()

# Run the training script
proc = subprocess.run(run_args, check=False)
print()
print(f"Training finished with return code: {proc.returncode}")
if proc.returncode == 0:
    print("✓ Training completed successfully!")
else:
    print("✗ Training encountered an error. Check the output above.")

### Troubleshooting & Tips

**Out of memory?** Reduce `batch_size` or set `gradient_accumulation_steps > 1`.

**Using multiple GPUs?** Comment out the cell above and run instead:
```bash
accelerate launch --num_processes=2 src_quantum/train.py --experiment_config <path>
```

**Want to watch training logs in real-time?** Check `<output_dir>/logs` or use Weights & Biases integration by setting `report_to: wandb` in the config.

## Part 5: Evaluation & Metrics

### Understanding Your Results

After training saves checkpoints, we evaluate them using:

- **FID (Fréchet Inception Distance)**: Measures realism of generated images
  - Lower is better (0 = identical to real data)
  - ~50 is typical for good translations
  
- **DINO (self-supervised vision features)**: Measures structure preservation
  - Compares semantic similarity between input and output
  - Lower is better (structure is preserved)

### Run Validation on Checkpoints

The validation script reuses the same FID/DINO computation from training callbacks but can be run independently on any checkpoint:

In [ ]:
import subprocess
import sys
from datetime import datetime
from pathlib import Path

# Build validation command
validation_args = [
    sys.executable,
    '-m',
    'src_quantum.val_cyclegan_turbo',
    '--experiment_config',
    str(EXPERIMENT_CONFIG),
]

output_dir = Path(training_cfg.output_dir)
if output_dir.exists() and list(output_dir.glob('checkpoints/*.pkl')):
    print("=" * 60)
    print("RUNNING VALIDATION")
    print("=" * 60)
    print(f"Evaluating checkpoints in: {output_dir}/checkpoints/")
    print()
    
    # Run validation
    proc = subprocess.run(validation_args, check=False)
    
    print()
    if proc.returncode == 0:
        print("✓ Validation completed!")
        print(f"Results saved to: {output_dir}/validation_*/")
        
        # Try to find and display the metrics CSV
        csv_files = list(output_dir.glob('*/validation_metrics.csv'))
        if csv_files:
            print(f"\nMetrics file: {csv_files[-1]}")
    else:
        print("✗ Validation encountered an error.")
else:
    print("⚠ No training checkpoints found yet.")
    print("Run the training cell above first, or update the output_dir path.")

### Interpreting Validation Output

The validation script generates:
- `validation_metrics.csv` → Table with FID and DINO scores for each checkpoint
- `samples_a2b/` & `samples_b2a/` → Generated images for visual inspection
- FID reference statistics → Used to compute FID distances

## Part 6: What's Next?

### Experiments to Try

1. **Disable quantum mode** for comparison:
   ```python
   custom_cfg = replace(quantum_cfg, quantum=False)
   ```
   Train a classical baseline to compare against quantum.

2. **Vary quantum circuit depth** (number of modes/photons):
   ```python
   custom_cfg = replace(quantum_cfg, num_modes=30, num_photons=5)
   ```
   Larger circuits = more expressivity but higher computational cost.

3. **Adjust loss weights**:
   ```python
   custom_cfg = replace(training_cfg, lambda_cycle=2.0, lambda_gan=0.25)
   ```
   Balance cycle consistency vs. adversarial realism.

4. **Use your own dataset**:
   - Place images in `dataset_folder/train_A/` and `dataset_folder/train_B/`
   - Update `dataset.dataset_folder` in the config

### Architecture Overview

```
Input Image A
    ↓
VAE Encoder → Classical Embeddings (C, H, W)
    ↓
Quantum Circuit (Boson Sampler) ← Trainable parameters
    ↓
Quantum Embeddings (processed via SLOS backend)
    ↓
U-Net Diffusion Decoder → Output Image B
```

### Key Files to Know

- `src_quantum/train.py` → Main training entry point
- `src_quantum/training/trainer.py` → Training loop implementation
- `src_quantum/config/defaults.py` → Configuration dataclasses
- `src_quantum/models/quantum_encoder.py` → Quantum circuit definition
- `src_quantum/models/cyclegan_turbo.py` → Classical diffusion model

### Further Reading

- Review `config/experiments/quantum_pretrained.yaml` for all available settings

Happy experimenting! 🚀